# CTI Attribution Claim Verification — Entailment Baseline Pipeline

This notebook implements **Week 2** of the research plan:
1. Retrieve evidence text for each attribution claim (from the source URLs gathered in Week 1)
2. Run a pretrained NLI (Natural Language Inference) entailment model to classify each claim as **True / False / Unsure**
3. Compare against the ground-truth labels from the Week 1 dataset
4. Build a **no-retrieval baseline** for comparison (shows the value of retrieval)
5. Compute accuracy, F1, and a confusion matrix — these are your baseline results for the paper

**How to use this notebook:**
1. Open this file in Google Colab (colab.research.google.com → File → Upload notebook)
2. Runtime → Run all
3. When prompted, upload your `attribution_claims_final_v2.csv` file (from Week 1)
4. Wait for it to finish (retrieval takes a few minutes, model download + inference a few more)
5. Results and a downloadable results CSV appear at the bottom

## Step 0 — Install packages

In [ ]:
!pip install -q transformers torch trafilatura requests scikit-learn pandas openpyxl

## Step 1 — Upload your Week 1 claims file

Upload `attribution_claims_final_v2.csv` (the file with all 85 claims + evidence_report links).

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
csv_name = list(uploaded.keys())[0]
claims = pd.read_csv(csv_name)
print(f"Loaded {len(claims)} claims")
claims.head()

## Step 2 — Retrieval: fetch evidence text for each claim

For claims with a real source URL, this fetches the article and extracts the main text (using `trafilatura`, which is good at stripping ads/navigation and keeping the article body). Claims with no URL (e.g. fabricated/unsure claims) get an empty premise, which is expected.

In [ ]:
import re
import trafilatura
import time

url_pattern = re.compile(r"\((https?://[^\)]+)\)")

def get_first_url(evidence_text):
    if not isinstance(evidence_text, str):
        return None
    m = url_pattern.search(evidence_text)
    return m.group(1) if m else None

def fetch_evidence_text(url, max_chars=3000):
    if not url:
        return ""
    try:
        downloaded = trafilatura.fetch_url(url, no_ssl=True)
        if downloaded is None:
            return ""
        text = trafilatura.extract(downloaded) or ""
        return text[:max_chars]
    except Exception as e:
        return ""

evidence_urls = claims["evidence_report"].apply(get_first_url)
evidence_texts = []
for i, url in enumerate(evidence_urls):
    text = fetch_evidence_text(url)
    evidence_texts.append(text)
    if (i + 1) % 10 == 0:
        print(f"  fetched {i+1}/{len(evidence_urls)}")

claims["evidence_text"] = evidence_texts
claims["evidence_text_len"] = claims["evidence_text"].str.len()
print("Done. Claims with usable retrieved text:", (claims["evidence_text_len"] > 100).sum(), "of", len(claims))
claims[["claim_id", "label", "evidence_text_len"]].head(10)

## Step 3 — Load the pretrained entailment (NLI) model

Using `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` — a strong, widely-used NLI model fine-tuned on MNLI + FEVER (fact verification) + ANLI. This is a well-cited choice for claim verification tasks, appropriate to cite in your Methodology section.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
print("Model loaded on", device)
print("Labels:", model.config.id2label)

## Step 4 — Entailment scoring function

For each claim: `premise` = retrieved evidence text, `hypothesis` = the claim text.
The model outputs probabilities for entailment / neutral / contradiction, which we map to True / Unsure / False.

In [ ]:
import torch.nn.functional as F

LABEL_MAP = {"entailment": "True", "neutral": "Unsure", "contradiction": "False"}

def score_entailment(premise, hypothesis):
    if not premise or len(premise.strip()) < 20:
        # No usable evidence retrieved -> cannot confirm -> Unsure
        return "Unsure", {"entailment": 0.0, "neutral": 1.0, "contradiction": 0.0}
    inputs = tokenizer(premise, hypothesis, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
    id2label = model.config.id2label
    label_probs = {id2label[i].lower(): float(probs[i]) for i in range(len(probs))}
    pred_raw = max(label_probs, key=label_probs.get)
    return LABEL_MAP.get(pred_raw, "Unsure"), label_probs

# Quick sanity test
test_premise = "MuddyWater has been observed using PowerShell scripts to stage local data before exfiltration."
test_hypothesis = "MuddyWater has used the technique Local Data Staging."
print(score_entailment(test_premise, test_hypothesis))

## Step 5 — Run the RETRIEVAL + ENTAILMENT system on all claims (main system)

In [ ]:
retrieval_preds = []
retrieval_probs = []
for i, row in claims.iterrows():
    pred, probs = score_entailment(row["evidence_text"], row["claim_text"])
    retrieval_preds.append(pred)
    retrieval_probs.append(probs)
    if (i + 1) % 10 == 0:
        print(f"  scored {i+1}/{len(claims)}")

claims["pred_with_retrieval"] = retrieval_preds
print("Done.")
claims[["claim_id", "label", "pred_with_retrieval"]].head(10)

## Step 6 — No-retrieval baseline (for comparison)

Same entailment model, but with NO retrieved evidence given — the model only sees the claim text itself as both premise and hypothesis context. This shows how much retrieval actually helps (the core argument of your paper).

In [ ]:
no_retrieval_preds = []
for i, row in claims.iterrows():
    # No evidence available -> premise is empty -> model cannot confirm -> forced to guess or default to Unsure
    pred, _ = score_entailment("", row["claim_text"])
    no_retrieval_preds.append(pred)

claims["pred_no_retrieval"] = no_retrieval_preds
print("Done.")
claims[["claim_id", "label", "pred_no_retrieval"]].head(10)

## Step 7 — Evaluate: accuracy, F1, confusion matrix

Ground-truth labels are collapsed to 3 classes for comparison:
- `True` = TRUE, TRUE-ALIAS
- `False` = FALSE-SWAP, FALSE-FAB
- `Unsure` = UNSURE

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

def collapse_label(label):
    label = str(label).upper()
    if label in ("TRUE", "TRUE-ALIAS"):
        return "True"
    if label in ("FALSE-SWAP", "FALSE-FAB", "FALSE-FABRICATED"):
        return "False"
    return "Unsure"

claims["gold_label"] = claims["label"].apply(collapse_label)

print("="*60)
print("SYSTEM: Retrieval + Entailment (main system)")
print("="*60)
print(f"Accuracy: {accuracy_score(claims['gold_label'], claims['pred_with_retrieval']):.3f}")
print(classification_report(claims["gold_label"], claims["pred_with_retrieval"], zero_division=0))
labels_order = ["True", "False", "Unsure"]
cm = confusion_matrix(claims["gold_label"], claims["pred_with_retrieval"], labels=labels_order)
print("Confusion matrix (rows=gold, cols=predicted), order:", labels_order)
print(cm)

print()
print("="*60)
print("BASELINE: No-Retrieval (claim text alone)")
print("="*60)
print(f"Accuracy: {accuracy_score(claims['gold_label'], claims['pred_no_retrieval']):.3f}")
print(classification_report(claims["gold_label"], claims["pred_no_retrieval"], zero_division=0))
cm2 = confusion_matrix(claims["gold_label"], claims["pred_no_retrieval"], labels=labels_order)
print("Confusion matrix (rows=gold, cols=predicted), order:", labels_order)
print(cm2)

## Step 8 — Save results (download this for your paper)

In [ ]:
out_cols = ["claim_id", "label", "gold_label", "claim_text", "evidence_text_len",
            "pred_with_retrieval", "pred_no_retrieval"]
results_df = claims[out_cols]
results_df.to_csv("week2_entailment_results.csv", index=False)

from google.colab import files
files.download("week2_entailment_results.csv")
print("Saved and downloading week2_entailment_results.csv")

## Notes for your paper

- **Model used**: `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` (DeBERTa-v3-base fine-tuned on MNLI, FEVER, and ANLI) — cite the model card / Laurer et al. paper on Hugging Face.
- **Retrieval method**: single-source retrieval directly from the cited evidence URL per claim (not a full corpus retriever) — appropriate to describe as a simplified/oracle retrieval setup for this baseline paper, with a note that full corpus-level retrieval (e.g. dense/hybrid retrieval over an indexed CTI report collection) is future work.
- **Claims with no retrievable evidence** (dead links, fabricated claims) default to "Unsure" — this is expected behavior and worth discussing in your Error Analysis section, not a bug.
- Compare the **with-retrieval vs no-retrieval accuracy gap** — this number is your key finding for the paper's Results section: it quantifies how much retrieval improves verification accuracy over the claim text alone.